# UTAU Auto OTO - Auto-Free 훈련 (Colab)

이 노트북은 Google Colab에서 Auto-Free(`autofree_v1`) 모델을 학습하기 위한 전용 노트북입니다.

실행 순서:
- Google Drive 마운트
- 프로젝트 스냅샷 동기화
- Python 의존성 설치
- 다중 보이스뱅크를 하나의 Auto-Free 학습 CSV로 빌드
- 모델 번들 1개 학습(`language + format`)
- 필요 시 산출물을 Drive로 복사


## 1) Drive 마운트 및 경로 설정

먼저 이 셀의 변수들을 실제 경로에 맞게 수정하세요.

`BANK_SPECS_JSON`의 권장 형식:

```json
[
  {
    "manual": "/abs/path/to/voicebank/oto.ini",
    "wav_dir": "/abs/path/to/voicebank",
    "tg_dir": "/abs/path/to/textgrid_or_empty",
    "token_map": "/abs/path/to/token_map.json_or_empty",
    "source_mode": "auto",
    "format_override": "cvvc",
    "voicebank_id": "optional_bank_id"
  }
]
```

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

USE_DRIVE_REPO_SNAPSHOT = True
PROJECT_GIT_URL = 'https://github.com/SODAsoo07/Auto_OTO.git'
PROJECT_BRANCH = 'codex/end2endMel'
DRIVE_REPO_SNAPSHOT = Path('/content/drive/MyDrive/UTAU_Auto_OTO_v3/Auto_OTO')

WORK_ROOT = Path('/content/auto_oto_colab')
PROJECT_ROOT = WORK_ROOT / 'Auto_OTO'

LANGUAGE = 'korean'  # korean | japanese
FORMAT_TYPE = 'cvvc'  # cv | cvc | cvvc | vcv | general

BANK_SPECS_JSON = Path('/content/drive/MyDrive/UTAU_Auto_OTO_v3/ml/configs/autofree_bank_specs_korean_cvvc.json')
DATASET_CSV = WORK_ROOT / 'artifacts' / f'dataset_autofree_{LANGUAGE}_{FORMAT_TYPE}.csv'
MODEL_OUT_DIR = WORK_ROOT / 'artifacts' / 'models' / LANGUAGE / FORMAT_TYPE / 'autofree_v1'

RUN_DATASET_BUILD = True
CLEAR_DATASET_BEFORE_BUILD = True
RUN_TRAIN = True

GROUP_COLUMN = 'voicebank_id'
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 50

COPY_ARTIFACTS_TO_DRIVE = True
DRIVE_ARTIFACT_DIR = Path('/content/drive/MyDrive/UTAU_Auto_OTO_v3/colab_artifacts/autofree')

WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_CSV.parent.mkdir(parents=True, exist_ok=True)
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'WORK_ROOT={WORK_ROOT}')
print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'BANK_SPECS_JSON={BANK_SPECS_JSON}')
print(f'DATASET_CSV={DATASET_CSV}')
print(f'MODEL_OUT_DIR={MODEL_OUT_DIR}')


## 2) 보조 함수

In [ ]:
import json
import os
import shutil
import subprocess
from typing import Any, Dict, List

def run_cmd(args, cwd=None, env=None):
    args = [str(a) for a in args]
    print('$', ' '.join(args))
    subprocess.run(args, cwd=cwd, env=env, check=True)

def run_json_cmd(args, cwd=None, env=None) -> Dict[str, Any]:
    args = [str(a) for a in args]
    print('$', ' '.join(args))
    cp = subprocess.run(args, cwd=cwd, env=env, check=True, capture_output=True, text=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    try:
        return json.loads(cp.stdout)
    except Exception:
        return {'stdout': cp.stdout, 'stderr': cp.stderr}

def sync_project_snapshot():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)

    if USE_DRIVE_REPO_SNAPSHOT:
        if not DRIVE_REPO_SNAPSHOT.exists():
            raise FileNotFoundError(f'Drive snapshot not found: {DRIVE_REPO_SNAPSHOT}')
        PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        run_cmd(['rsync', '-a', '--delete', '--exclude', '.git', f'{DRIVE_REPO_SNAPSHOT}/', f'{PROJECT_ROOT}/'])
    else:
        run_cmd(['git', 'clone', '--depth', '1', '--branch', PROJECT_BRANCH, PROJECT_GIT_URL, str(PROJECT_ROOT)])

def load_bank_specs(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f'BANK_SPECS_JSON not found: {path}')
    payload = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(payload, list):
        raise ValueError('BANK_SPECS_JSON must be a JSON list')
    out = []
    for i, item in enumerate(payload):
        if not isinstance(item, dict):
            raise ValueError(f'bank spec #{i} is not an object')
        manual = str(item.get('manual', '')).strip()
        wav_dir = str(item.get('wav_dir', '')).strip()
        if not manual or not wav_dir:
            raise ValueError(f'bank spec #{i} must have manual and wav_dir')
        out.append(item)
    return out


## 3) 프로젝트 동기화 및 의존성 설치

In [ ]:
sync_project_snapshot()
os.chdir(PROJECT_ROOT)
print('cwd =', PROJECT_ROOT)

run_cmd(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run_cmd([
    'python', '-m', 'pip', 'install', '-q',
    '-r', str(PROJECT_ROOT / 'requirements.txt'),
    '-r', str(PROJECT_ROOT / 'requirements-ml.txt')
])

os.environ['PYTHONUTF8'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'


## 4) Auto-Free 학습용 데이터셋 CSV 생성 (다중 보이스뱅크)

- `source_mode`: `auto` / `textgrid` / `audio_only`
- `RUN_DATASET_BUILD=False`이면 이 단계는 건너뛰고 기존 `DATASET_CSV`를 사용합니다.

In [ ]:
if RUN_DATASET_BUILD:
    specs = load_bank_specs(BANK_SPECS_JSON)
    print(f'Loaded bank specs: {len(specs)}')

    if CLEAR_DATASET_BEFORE_BUILD and DATASET_CSV.exists():
        DATASET_CSV.unlink()
        print(f'Removed old dataset: {DATASET_CSV}')

    script_path = PROJECT_ROOT / 'ml' / 'scripts' / 'autofree' / 'build_dataset.py'
    build_reports = []

    for idx, spec in enumerate(specs):
        args = [
            'python', '-X', 'utf8', str(script_path),
            '--lang', LANGUAGE,
            '--manual', spec['manual'],
            '--wav-dir', spec['wav_dir'],
            '--out', str(DATASET_CSV),
        ]

        tg_dir = str(spec.get('tg_dir', '') or '').strip()
        if tg_dir:
            args += ['--tg-dir', tg_dir]

        token_map = str(spec.get('token_map', '') or '').strip()
        if token_map:
            args += ['--token-map', token_map]

        source_mode = str(spec.get('source_mode', 'auto') or 'auto').strip()
        format_override = str(spec.get('format_override', FORMAT_TYPE) or FORMAT_TYPE).strip()
        voicebank_id = str(spec.get('voicebank_id', '') or '').strip()

        args += ['--source-mode', source_mode]
        if format_override:
            args += ['--format-override', format_override]
        if voicebank_id:
            args += ['--voicebank-id', voicebank_id]
        if idx > 0 or DATASET_CSV.exists():
            args += ['--append']

        report = run_json_cmd(args, cwd=PROJECT_ROOT)
        report['bank_index'] = idx
        report['manual'] = spec.get('manual')
        build_reports.append(report)

    print(f'Build complete. report_count={len(build_reports)} dataset={DATASET_CSV}')
else:
    print('RUN_DATASET_BUILD=False -> skip build step')

if not DATASET_CSV.exists():
    raise FileNotFoundError(f'Dataset CSV not found: {DATASET_CSV}')
print('Dataset CSV ready:', DATASET_CSV)


## 5) 데이터셋 기본 점검

In [ ]:
import pandas as pd

df = pd.read_csv(DATASET_CSV)
print('rows =', len(df))
print('columns =', len(df.columns))

for col in ['language', 'format_type', 'source_mode', 'voicebank_id']:
    if col in df.columns:
        print(f'--- {col} top values ---')
        print(df[col].value_counts(dropna=False).head(20))

required_targets = [
    'target_offset_ms',
    'target_cons_ms',
    'target_cutoff_abs_ms',
    'target_pre_ms',
    'target_ovl_ms',
]
missing = [c for c in required_targets if c not in df.columns]
if missing:
    raise RuntimeError(f'missing required target columns: {missing}')

print('Target columns OK')


## 6) Auto-Free 모델 번들 학습

In [ ]:
if RUN_TRAIN:
    train_script = PROJECT_ROOT / 'ml' / 'scripts' / 'autofree' / 'train.py'
    train_args = [
        'python', '-X', 'utf8', str(train_script),
        '--lang', LANGUAGE,
        '--format', FORMAT_TYPE,
        '--dataset', str(DATASET_CSV),
        '--out-dir', str(MODEL_OUT_DIR),
        '--group-column', GROUP_COLUMN,
        '--num-boost-round', str(NUM_BOOST_ROUND),
        '--early-stopping-rounds', str(EARLY_STOPPING_ROUNDS),
    ]
    train_meta = run_json_cmd(train_args, cwd=PROJECT_ROOT)
    print('Training finished')
else:
    print('RUN_TRAIN=False -> skip train step')

meta_path = MODEL_OUT_DIR / 'model_meta.json'
eval_path = MODEL_OUT_DIR / 'eval_summary.json'
if meta_path.exists():
    print('model_meta.json')
    print(meta_path.read_text(encoding='utf-8'))
else:
    print('model_meta.json not found')

if eval_path.exists():
    print('eval_summary.json')
    print(eval_path.read_text(encoding='utf-8'))
else:
    print('eval_summary.json not found')


## 7) 산출물 Drive 복사 (선택)

In [ ]:
if COPY_ARTIFACTS_TO_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    dst_root = DRIVE_ARTIFACT_DIR / LANGUAGE / FORMAT_TYPE / 'autofree_v1'
    dst_root.mkdir(parents=True, exist_ok=True)

    run_cmd(['rsync', '-a', str(MODEL_OUT_DIR) + '/', str(dst_root) + '/'])
    run_cmd(['rsync', '-a', str(DATASET_CSV), str(dst_root / DATASET_CSV.name)])
    print('Artifacts copied to:', dst_root)
else:
    print('COPY_ARTIFACTS_TO_DRIVE=False -> skip copy')
